# 🎓 inVision U — Training NLP Models v3
**Стратегия:**
- 🇬🇧 EN → RoBERTa fine-tune (без изменений, уже есть `roberta_en.pkl`)
- 🇷🇺🇰🇿 RU + KZ → mpnet + SVR на **всех 1685 записях** + lang как признак

> Если `roberta_en.pkl` уже скачан — секцию EN можно пропустить, запускай с секции 5.

Runtime → Change runtime type → **T4 GPU**


## 1. Install

In [1]:
%%capture
!pip install -U transformers sentence-transformers scikit-learn shap


## 2. Imports

In [2]:
import json, pickle, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")


Device: cuda
GPU: Tesla T4


## 3. Upload dataset_final.json

In [3]:
from google.colab import files
print("Upload dataset_final.json")
uploaded = files.upload()

with open("dataset_final.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

df = pd.DataFrame(dataset)
print(f"Total: {len(df)}")
print(df['lang'].value_counts())


Upload dataset_final.json


Saving dataset_final.json to dataset_final.json
Total: 1685
lang
en    1185
ru     280
kz     220
Name: count, dtype: int64


## 4. Config

In [4]:
QUALITY_KEYS = [
    "leadership", "motivation", "perseverance",
    "teamwork", "goal_pursuit", "intrinsic_motivation", "prosocial_purpose"
]

# EN fine-tune (из v2, без изменений)
EN_MODEL_NAME = "roberta-base"
EN_MAX_LEN    = 256
EN_BATCH_SIZE = 16
EN_EPOCHS     = 5
EN_LR         = 2e-5
EN_OUT_FILE   = "roberta_en.pkl"

# Общая модель RU+KZ
ST_MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"
ST_OUT_RU     = "rubert_ru.pkl"
ST_OUT_KZ     = "mbert_kz.pkl"
ST_OUT_SHARED = "shared_rukz.pkl"   # сама общая модель


## 5. EN — RoBERTa Fine-tune
> Если `roberta_en.pkl` уже есть — пропусти эту секцию и иди к секции 6.


In [5]:
class EssayDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True,
            max_length=self.max_len, padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels": torch.tensor(self.labels[idx], dtype=torch.float)}

class RoBERTaRegressor(nn.Module):
    def __init__(self, model_name, num_outputs):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden       = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.head    = nn.Sequential(
            nn.Linear(hidden, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, num_outputs))
    def forward(self, input_ids, attention_mask):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]
        return self.head(self.dropout(pooled))

def train_en():
    print(f"\n{'='*55}\n  Fine-tuning: {EN_MODEL_NAME}  (lang=en)\n{'='*55}")
    subset = df[df["lang"] == "en"].reset_index(drop=True)
    texts  = subset["text"].tolist()
    labels = np.array([[r[k] for k in QUALITY_KEYS]
                        for r in subset["scores"]], dtype=np.float32)
    labels_norm = (labels - 1) / 9.0
    X_train_t, X_test_t, y_train, y_test = train_test_split(
        texts, labels_norm, test_size=0.15, random_state=42)
    tokenizer = AutoTokenizer.from_pretrained(EN_MODEL_NAME)
    train_dl  = DataLoader(EssayDataset(X_train_t, y_train, tokenizer, EN_MAX_LEN),
                           batch_size=EN_BATCH_SIZE, shuffle=True)
    test_dl   = DataLoader(EssayDataset(X_test_t, y_test, tokenizer, EN_MAX_LEN),
                           batch_size=EN_BATCH_SIZE)
    model     = RoBERTaRegressor(EN_MODEL_NAME, len(QUALITY_KEYS)).to(device)
    optimizer = AdamW(model.parameters(), lr=EN_LR, weight_decay=0.01)
    total_steps = len(train_dl) * EN_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=total_steps//10,
        num_training_steps=total_steps)
    criterion = nn.MSELoss()
    print(f"\nTraining {EN_EPOCHS} epochs...")
    for epoch in range(EN_EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_dl:
            optimizer.zero_grad()
            preds = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            loss  = criterion(preds, batch["labels"].to(device))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1}/{EN_EPOCHS}  loss={total_loss/len(train_dl):.4f}")
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_dl:
            preds = model(batch["input_ids"].to(device),
                          batch["attention_mask"].to(device)).cpu().numpy()
            all_preds.append(preds); all_labels.append(batch["labels"].numpy())
    y_pred = np.vstack(all_preds) * 9 + 1
    y_true = np.vstack(all_labels) * 9 + 1
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred, multioutput="uniform_average")
    print(f"\n📊 EN  MAE={mae:.4f}  R²={r2:.4f}")
    artifact = {"model_type": "finetune", "model_name": EN_MODEL_NAME, "lang": "en",
                "tokenizer": tokenizer, "model": model.cpu().state_dict(),
                "model_config": {"hidden": model.bert.config.hidden_size,
                                 "num_outputs": len(QUALITY_KEYS)},
                "quality_keys": QUALITY_KEYS, "metrics": {"mae": float(mae), "r2": float(r2)}}
    with open(EN_OUT_FILE, "wb") as f: pickle.dump(artifact, f)
    print(f"✅ Saved → {EN_OUT_FILE}")
    return artifact

en_artifact = train_en()



  Fine-tuning: roberta-base  (lang=en)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training 5 epochs...
  Epoch 1/5  loss=0.0617
  Epoch 2/5  loss=0.0264
  Epoch 3/5  loss=0.0211
  Epoch 4/5  loss=0.0190
  Epoch 5/5  loss=0.0171

📊 EN  MAE=0.9061  R²=0.5590
✅ Saved → roberta_en.pkl


## 6. Общая модель RU + KZ (все 1685 записей + lang feature)

In [6]:
def train_shared_rukz():
    print(f"\n{'='*55}")
    print(f"  Shared model: {ST_MODEL_NAME}  (ALL langs)")
    print(f"{'='*55}")

    # ── Все данные ───────────────────────────────────────────────────────
    texts  = df["text"].tolist()
    langs  = df["lang"].tolist()
    labels = np.array([[r[k] for k in QUALITY_KEYS]
                        for r in df["scores"]], dtype=np.float32)

    print(f"Total records: {len(texts)}")
    print(f"Lang dist: {dict(Counter(langs))}")

    # ── Sentence embeddings ──────────────────────────────────────────────
    print("\nGenerating embeddings for ALL records...")
    st_model = SentenceTransformer(ST_MODEL_NAME, device=str(device))
    X_emb = st_model.encode(texts, batch_size=32, show_progress_bar=True,
                             convert_to_numpy=True, normalize_embeddings=True)
    print(f"Embedding shape: {X_emb.shape}")

    # ── Lang one-hot feature [is_en, is_ru, is_kz] ──────────────────────
    lang_map = {"en": [1,0,0], "ru": [0,1,0], "kz": [0,0,1]}
    X_lang = np.array([lang_map[l] for l in langs], dtype=np.float32)

    # ── Concatenate: embeddings + lang ───────────────────────────────────
    X = np.hstack([X_emb, X_lang])
    print(f"Final feature shape (emb + lang): {X.shape}")

    # ── Scale ────────────────────────────────────────────────────────────
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ── Train/test split — стратифицированно по языку ────────────────────
    X_train, X_test, y_train, y_test, langs_train, langs_test = train_test_split(
        X_scaled, labels, langs, test_size=0.15, random_state=42, stratify=langs)
    print(f"Train: {len(X_train)} | Test: {len(X_test)}")
    print(f"Test lang dist: {dict(Counter(langs_test))}")

    # ── SVR ──────────────────────────────────────────────────────────────
    print("\nFitting SVR (RBF kernel)...")
    clf = MultiOutputRegressor(
        SVR(kernel="rbf", C=10.0, gamma="scale", epsilon=0.3),
        n_jobs=-1)
    clf.fit(X_train, y_train)

    # ── Evaluate overall ─────────────────────────────────────────────────
    y_pred = clf.predict(X_test)
    mae_all = mean_absolute_error(y_test, y_pred)
    r2_all  = r2_score(y_test, y_pred, multioutput="uniform_average")
    print(f"\n📊 Overall  MAE={mae_all:.4f}  R²={r2_all:.4f}")

    # ── Evaluate per language ─────────────────────────────────────────────
    metrics = {}
    for lang in ["en", "ru", "kz"]:
        idx = [i for i, l in enumerate(langs_test) if l == lang]
        if not idx: continue
        m = mean_absolute_error(y_test[idx], y_pred[idx])
        r = r2_score(y_test[idx], y_pred[idx], multioutput="uniform_average")
        metrics[lang] = {"mae": float(m), "r2": float(r)}
        print(f"  {lang.upper():<4} MAE={m:.4f}  R²={r:.4f}  (n_test={len(idx)})")

    print("\nPer quality:")
    for i, q in enumerate(QUALITY_KEYS):
        m = mean_absolute_error(y_test[:,i], y_pred[:,i])
        print(f"  {q:<25} MAE={m:.3f}")

    # ── Save shared model ─────────────────────────────────────────────────
    shared_artifact = {
        "model_type":   "sentence_transformers_shared",
        "model_name":   ST_MODEL_NAME,
        "lang":         "all",
        "st_model":     st_model,
        "scaler":       scaler,
        "classifier":   clf,
        "quality_keys": QUALITY_KEYS,
        "lang_map":     lang_map,
        "metrics":      metrics,
    }
    with open(ST_OUT_SHARED, "wb") as f: pickle.dump(shared_artifact, f)
    print(f"\n✅ Saved shared → {ST_OUT_SHARED}")

    # ── Создаём RU и KZ .pkl как обёртки над shared ───────────────────────
    for lang, out_file in [("ru", ST_OUT_RU), ("kz", ST_OUT_KZ)]:
        artifact = {**shared_artifact, "lang": lang}
        with open(out_file, "wb") as f: pickle.dump(artifact, f)
        size = Path(out_file).stat().st_size / 1e6
        print(f"✅ Saved → {out_file}  ({size:.1f} MB)")

    return shared_artifact

shared_artifact = train_shared_rukz()



  Shared model: paraphrase-multilingual-mpnet-base-v2  (ALL langs)
Total records: 1685
Lang dist: {'en': 1185, 'ru': 280, 'kz': 220}

Generating embeddings for ALL records...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

Embedding shape: (1685, 768)
Final feature shape (emb + lang): (1685, 771)
Train: 1432 | Test: 253
Test lang dist: {'ru': 42, 'en': 178, 'kz': 33}

Fitting SVR (RBF kernel)...

📊 Overall  MAE=1.0928  R²=0.4917
  EN   MAE=0.9250  R²=0.5741  (n_test=178)
  RU   MAE=1.4452  R²=0.0845  (n_test=42)
  KZ   MAE=1.5493  R²=0.0403  (n_test=33)

Per quality:
  leadership                MAE=1.056
  motivation                MAE=0.986
  perseverance              MAE=1.106
  teamwork                  MAE=1.089
  goal_pursuit              MAE=0.985
  intrinsic_motivation      MAE=1.181
  prosocial_purpose         MAE=1.246

✅ Saved shared → shared_rukz.pkl
✅ Saved → rubert_ru.pkl  (1177.6 MB)
✅ Saved → mbert_kz.pkl  (1177.6 MB)


## 7. Summary

In [7]:
print("\n🎉 All models trained!")
m = en_artifact['metrics']
print(f"  EN  MAE={m['mae']:.4f}  R²={m['r2']:.4f}  (RoBERTa fine-tune)")
for lang in ["ru", "kz"]:
    m = shared_artifact['metrics'].get(lang, {})
    print(f"  {lang.upper()}  MAE={m.get('mae','?'):.4f}  R²={m.get('r2','?'):.4f}  (shared mpnet+SVR)")



🎉 All models trained!
  EN  MAE=0.9061  R²=0.5590  (RoBERTa fine-tune)
  RU  MAE=1.4452  R²=0.0845  (shared mpnet+SVR)
  KZ  MAE=1.5493  R²=0.0403  (shared mpnet+SVR)


## 8. Download .pkl

In [8]:
from google.colab import files
for f in [EN_OUT_FILE, ST_OUT_RU, ST_OUT_KZ]:
    print(f"Downloading {f}...")
    files.download(f)
print("\nПомести файлы в backend/models_trained/")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Помести файлы в backend/models_trained/


## 9. Smoke test

In [9]:
def smoke_finetune(pkl_path, text):
    with open(pkl_path, "rb") as f: art = pickle.load(f)
    cfg = art["model_config"]
    model = RoBERTaRegressor(art["model_name"], cfg["num_outputs"])
    model.load_state_dict(art["model"]); model.eval().to(device)
    enc = art["tokenizer"](text, return_tensors="pt", truncation=True,
                           max_length=256, padding="max_length").to(device)
    with torch.no_grad():
        pred = model(enc["input_ids"], enc["attention_mask"]).cpu().numpy()[0] * 9 + 1
    print(f"\n🔍 {pkl_path}")
    for k, v in zip(art["quality_keys"], pred): print(f"  {k:<25} {v:.1f}/10")

def smoke_shared(pkl_path, text, lang):
    with open(pkl_path, "rb") as f: art = pickle.load(f)
    X_emb = art["st_model"].encode([text], normalize_embeddings=True)
    X_lang = np.array([art["lang_map"][lang]], dtype=np.float32)
    X = np.hstack([X_emb, X_lang])
    X_scaled = art["scaler"].transform(X)
    pred = art["classifier"].predict(X_scaled)[0]
    print(f"\n🔍 {pkl_path} (lang={lang})")
    for k, v in zip(art["quality_keys"], pred): print(f"  {k:<25} {v:.1f}/10")

smoke_finetune(EN_OUT_FILE,
    "I founded a student club and led 20 volunteers to clean our local river.")
smoke_shared(ST_OUT_RU,
    "Я организовал волонтёрский отряд и провёл три экологические акции.", "ru")
smoke_shared(ST_OUT_KZ,
    "Мен мектепте волонтерлер тобын құрдым және үш экологиялық акция өткіздім.", "kz")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🔍 roberta_en.pkl
  leadership                6.5/10
  motivation                6.3/10
  perseverance              5.1/10
  teamwork                  5.3/10
  goal_pursuit              4.6/10
  intrinsic_motivation      5.3/10
  prosocial_purpose         6.0/10

🔍 rubert_ru.pkl (lang=ru)
  leadership                5.6/10
  motivation                4.9/10
  perseverance              3.5/10
  teamwork                  4.0/10
  goal_pursuit              4.0/10
  intrinsic_motivation      4.3/10
  prosocial_purpose         3.9/10

🔍 mbert_kz.pkl (lang=kz)
  leadership                6.3/10
  motivation                4.7/10
  perseverance              3.6/10
  teamwork                  3.9/10
  goal_pursuit              3.9/10
  intrinsic_motivation      3.7/10
  prosocial_purpose         3.8/10


---
## ✅ Done

| Файл | Подход | Ожидаемый R² |
|---|---|---|
| `roberta_en.pkl` | RoBERTa fine-tune | ~0.55 |
| `rubert_ru.pkl` | shared mpnet + SVR + lang feature | ~0.35–0.45 |
| `mbert_kz.pkl` | shared mpnet + SVR + lang feature | ~0.35–0.45 |

Положи все три в `backend/models_trained/`
